1. **Tool-gating experiment** — Take 50 queries from Bitext (you already have it). For each, run: (a) single-shot LLM, (b) LLM + 3 tools (`lookup_order`, `check_policy`, `escalate`).

LLM without TOOLS

In [28]:
from langchain_ollama import ChatOllama

In [29]:
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0.7
)

result = llm.invoke("Where is my order #ORDNO1?")
print(result.content)

I don't have any information about an order with the number "ORDNO1". I'm a large language model, I don't have the ability to keep track of orders or store any information about individual customers or orders.

If you're trying to track down an order, I recommend contacting the company or merchant that you placed the order with directly. They should be able to provide you with the most up-to-date information about the status of your order.

If you're trying to locate an order within a specific system or application, I may be able to help you troubleshoot or provide guidance on how to find the information you're looking for. Can you provide more context or details about where you're trying to find the order?


LLM with TOOLS

In [30]:
from langchain_core.tools import tool
import pandas as pd

@tool
def lookupOrder(orderNo:str) -> str:
    """Look up order status by order ID. Order IDs look like #ORDNO1, #ORDNO2, etc."""
    df = pd.read_csv("order_mock_data.csv")
    if not orderNo.startswith("#"):
        orderNo = "#" + orderNo
    row = df[df["id"] == orderNo]
    if row.empty:
        return f"No order found with ID {orderNo}. Please ask the customer to verify the order number."
    data = f"OrderNo: {row['id'].iloc[0]}, Name: {row['name'].iloc[0]}, InvoiceNo: {row['invoice_number'].iloc[0]}, Status: {row['status'].iloc[0]}, Items: {row['items'].iloc[0]}"
    return data

lookupOrder.invoke("#ORDNO1")

'OrderNo: #ORDNO1, Name: Matthew Brown, InvoiceNo: INV-280587, Status: Ordered, Items: Desk Lamp, Ergonomic Wrist Rest, Power Bank 10000mAh'

In [31]:
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document


def data_injestion(filepath:str):
    loader = PyPDFLoader(filepath)
    pages = loader.load()
    text_spliiter = RecursiveCharacterTextSplitter(
        chunk_size = 1000, 
        chunk_overlap = 200
    )
    chunks = text_spliiter.split_documents(pages)
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    vectordb = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory='./chroma.db'
    )
    retriever = vectordb.as_retriever(search_kwargs={"k":3})
    return retriever

retriever = data_injestion("sample_policy.pdf")

In [32]:
@tool
def check_policy(query:str) -> str:
    """Query the company policy on order issues and queries on product damages and so on."""
    context_chunks = retriever.invoke(query)
    return "\n\n".join([c.page_content for c in context_chunks])

In [33]:
tools = [
    lookupOrder,
    check_policy,
]

llm = ChatOllama(model = "llama3.2:3b")
llm_with_tool = llm.bind_tools(tools)

queries = [
    "Where is my order #ORDNO5? Can you check the status?",  # → lookupOrder
    "My product arrived damaged. What should I do?",          # → check_policy
    "I'm furious, my package never arrived and I want to sue your company!"  # → escalation
]

In [34]:
from langchain_core.messages import HumanMessage, ToolMessage

tool_map = {"lookupOrder": lookupOrder, "check_policy": check_policy}

def run_agent(query: str):
    messages = [HumanMessage(content=query)]
    
    while True:
        response = llm_with_tool.invoke(messages)
        messages.append(response)
        
        if not response.tool_calls:
            return response.content  # final answer
        
        # LLM wants to call tools — execute each one
        for tool_call in response.tool_calls:
            tool = tool_map[tool_call["name"]]
            result = tool.invoke(tool_call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))


In [35]:
print(run_agent("Where is my order #ORDNO5?"))


Your order #ORDNO5 is currently in the shipping stage. It includes the following items:

* A Fitness Tracker
* A Desk Lamp

You can expect to receive your order within the next 3-5 business days. If you have any questions or concerns about your order, you can contact our customer service team at [insert contact information].
